# Case definitions: burns and head injuries

**Nicholas Soulakis, PhD** · October 2026

**Summary.** Surveillance starts by deciding what counts as a case. This notebook applies two case definitions, burns and head injuries, to every published year of the National Electronic Injury Surveillance System (NEISS), which records **consumer product–related injuries** treated in a national sample of hospital emergency departments, checks the data the definitions depend on, and reports sampled cases and national estimates with their uncertainty.

*Adapted from Methods in Health and Biomedical Informatics, taught by the author as HSIP 441 at Northwestern University Feinberg School of Medicine and MSBI 33100 at the University of Chicago. Python implementation, 2026.*

## Why it matters

A surveillance system can only watch for what it can define. A case definition turns a question ("are burns rising?") into a rule applied to every record. NEISS is a probability sample of hospital emergency departments, so the rule has to be paired with the sample design: a count of sampled visits says what the sample saw; a national estimate needs the survey weights and its own measure of uncertainty.

In [1]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("../utils").exists():
    os.system("git clone -q https://github.com/nsoulakis/public_health_surveillance")
    os.chdir("public_health_surveillance/python/notebooks")
sys.path.insert(0, "..")

In [2]:
import pandas as pd
from utils import neiss, survey

df = neiss.code_cases(neiss.read_years())
formats = neiss.read_formats()
year = df["trmt_date"].dt.year
print(f"NEISS {year.min()}-{year.max()}: {len(df):,} sampled visits from published CPSC files")

NEISS 2014-2025: 4,292,396 sampled visits from published CPSC files


## Check the data first

Four checks, before any definition is trusted:

1. **Codes outside CPSC's formats.** A code the formats file does not define is counted, not silently kept or dropped.
2. **Which fields exist each year.** NEISS added a second diagnosis and body part to its files; a definition that uses either field finds more cases in the years the second field is filled.
3. **Hospitals reporting each year.** Sampled counts move when the number of reporting hospitals changes.

In [3]:
checks = neiss.validity_report(df, formats)
odd = checks[checks["not_in_formats"] > 0]
print("Codes not in CPSC's formats:" if len(odd) else "Every code is in CPSC's formats.")
odd

Codes not in CPSC's formats:


,year,field,blank,not_in_formats
7,2021,sex,0,11
8,2022,sex,0,33
9,2023,sex,0,157


In [4]:
fields = pd.DataFrame({
    "visits": year.value_counts().sort_index(),
    "share_with_second_body_part": df["bdpt2"].notna().groupby(year).mean().round(3),
    "reporting_hospitals": df.groupby(year)["psu"].nunique(),
})
fields

,visits,share_with_second_body_part,reporting_hospitals
trmt_date,,,
2014,367493,0.000,86
2015,359129,0.000,82
2016,375196,0.000,84
2017,386906,0.000,89
2018,361667,0.000,88
2019,358715,0.196,84
2020,309370,0.212,82
2021,340442,0.212,80
2022,323343,0.224,78


4. **The sample design each year.** CPSC assigns each hospital to a stratum and sets the weights; when the design changes, estimates can jump with no change in injuries. The table shows hospitals per stratum, the mean weight per visit, and how many hospitals changed stratum from the year before.

In [5]:
by_stratum = df.groupby([year, "stratum"])["psu"].nunique().unstack(fill_value=0)
stratum_of = df.groupby([year, "psu"])["stratum"].first().unstack()
moved = (stratum_of != stratum_of.shift(1)) & stratum_of.notna() & stratum_of.shift(1).notna()
design = by_stratum.assign(mean_weight=df.groupby(year)["wt"].mean().round(1),
                           hospitals_changing_stratum=moved.sum(axis=1).astype(int))
design.index.name = "year"
design

stratum,C,L,M,S,V,mean_weight,hospitals_changing_stratum
year,,,,,,,
2014,7,7,11,38,23,37.7,0
2015,8,7,9,36,22,39.4,0
2016,8,8,10,36,22,38.2,0
2017,8,8,11,39,23,38.1,0
2018,8,9,11,39,21,38.6,0
2019,8,9,10,36,21,37.5,0
2020,8,9,10,35,20,35.5,0
2021,8,9,10,32,21,34.5,0
2022,8,9,10,31,20,39.2,0


## The definitions

- **Adults:** older than 18. NEISS records children under 2 in completed months as 200 + months (201–223); those codes are converted to years first, and an unknown age, recorded as 0, is left out.¹
- **Burn:** a burn diagnosis (electrical, not specified, scald, chemical, thermal) in either diagnosis field. Radiation burns (diagnosis 73, mostly sunburn and tanning) are not part of this definition.
- **Head injury:** the head as the body part in either body-part field, whatever the diagnosis. A body part coded "not recorded" (87) is missing data, not a "no".

Each visit is a case, a non-case, or unclassifiable (no recorded diagnosis, or no recorded body part). Unclassifiable visits are counted and kept out of every numerator and denominator. Every published year is used, and every recorded diagnosis: the second fields are part of the data from the year NEISS fills them.

¹ NEISS Coding Manual (CPSC Only "Non-Trauma"), January 2025, p. 11. Codes are assigned by trained NEISS coders from the clinical record.

In [6]:
adult = df["age_years"] > neiss.ADULT_AGE
print(f"Adult visits: {adult.sum():,}; unknown age: {df['age_years'].isna().sum():,}")
neiss.case_counts(df[adult])

Adult visits: 2,435,187; unknown age: 361


,definition,cases,non_cases,unclassifiable
0,burn,38963,2396224,0
1,head_injury,406262,1978313,50612


## Sampled cases are not incidence

The counts above are **sampled visits**: what the reporting hospitals recorded. NEISS covers consumer product–related injuries only, so every estimate below is of **consumer product–related emergency department visits**, not all emergency visits for burns or head injuries. A national estimate adds the survey weights, and its uncertainty comes from the design: hospitals within strata, with every reporting hospital included even when it had no cases that year.² CPSC treats an estimate as unstable when it rests on fewer than 20 records, is below 1,200, or has a coefficient of variation (CV) above 33%; unstable estimates are marked and their figures withheld.³

² Schroeder T, Ault K. *The NEISS Sample (Design and Implementation) 1997 to Present.* U.S. Consumer Product Safety Commission, 2001 (estimate eq. 3; variance p. 13 and Appendix 1). The public file's PSU is used as the hospital. A hospital reporting in any month of the year is counted; Schroeder & Ault count those reporting in a majority of months, so these standard errors are slightly conservative.
³ CPSC, NEISS web estimates: cpsc.gov/cgibin/NEISSQuery/webestimates.html. 95% CI = estimate ± 1.96 × estimate × CV.

In [7]:
def table(definition):
    e = survey.estimate(df, definition, within=adult)
    shown = e.copy()
    for col in ["estimate", "ci_low", "ci_high"]:
        shown[col] = e[col].round(-2).astype(int).astype(str).where(e["stable"], "unstable")
    shown["cv"] = e["cv"].round(3).astype(str).where(e["stable"], "")
    return shown[["year", "sampled_cases", "estimate", "cv", "ci_low", "ci_high"]]

table("burn")

,year,sampled_cases,estimate,cv,ci_low,ci_high
0,2014,3247,127700,0.075,108900,146500
1,2015,3263,131100,0.079,110900,151400
2,2016,3336,141500,0.085,118000,165100
3,2017,3348,144600,0.081,121600,167500
4,2018,3119,131900,0.08,111400,152500
5,2019,3205,128400,0.086,106800,150000
6,2020,3141,119000,0.106,94300,143700
7,2021,3221,117500,0.095,95700,139300
8,2022,2974,123600,0.114,95900,151300
9,2023,3184,126800,0.102,101500,152000


In [8]:
table("head_injury")

,year,sampled_cases,estimate,cv,ci_low,ci_high
0,2014,27284,1092300,0.094,890100,1294500
1,2015,27454,1186400,0.115,918900,1453900
2,2016,28469,1214400,0.108,957400,1471400
3,2017,29724,1287300,0.095,1048300,1526400
4,2018,29984,1338800,0.101,1073300,1604200
5,2019,34563,1544800,0.097,1252500,1837100
6,2020,31740,1371200,0.104,1092400,1650000
7,2021,35492,1455900,0.096,1181500,1730400
8,2022,35429,1661900,0.117,1281600,2042200
9,2023,37535,1630900,0.105,1295800,1966100


## What the second field adds

The share of visits with a second body part, above, is zero until NEISS begins filling the field. The table below counts the adult cases found only through the second field. Those cases are real recorded diagnoses, so they are kept; the jump they create is a change in what NEISS records, not a change in injuries, and any trend read across it has to say so.

In [9]:
a = df[adult]
second_only = pd.DataFrame({
    "burn": ((a["burn"] == 1) & ~a["diag"].isin(neiss.BURN_DIAG)).groupby(a["trmt_date"].dt.year).sum(),
    "head_injury": ((a["head_injury"] == 1) & ~a["bdpt"].isin(neiss.HEAD_BDPT)).groupby(a["trmt_date"].dt.year).sum(),
})
second_only

,burn,head_injury
trmt_date,,
2014,0,0
2015,0,0
2016,0,0
2017,0,0
2018,0,0
2019,95,2834
2020,126,2981
2021,135,3461
2022,164,3343


## Limits

- Sampled counts depend on how many hospitals report; compare years with the reporting hospitals above in view.
- Trends that cross the year the second fields begin mix two recording practices.
- Variance treats each public-file PSU as one hospital.
- Estimates that cross a change in the sample design (strata or weights, above) compare different designs; read a jump that coincides with one as a design change first.

## Differences from the original course programs

This notebook follows the author's course programs with these deliberate changes:
- Ages recorded in months are converted before the adult filter is applied (the original filtered first, which kept children under 2).
- Cases are counted as visits meeting the definition (the original summed the 1/2 case codes).
- Head injury is the head as body part; the original course example also required a sports product.
- Every published year from 2014 is used, with the second diagnosis and body part wherever NEISS filled them.

## Going further

- Add a third definition, for example falls on stairs (a product code plus a diagnosis), and check its stability year by year.
- Search the narratives for a term the codes miss, and count how many cases the codes alone would have found.
- Next notebook: daily signals from these definitions.